In [9]:
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import os 

In [10]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key = openai_api_key)

In [11]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

def message_gpt(prompt):
    messages = [{"role":"system", "content":system_message}, {"role":"user", "content":prompt}]
    response = client.chat.completions.create(model="gpt-4.1-mini", messages=messages, stream=True)
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [12]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")


In [13]:
view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7861


KeyboardInterrupt: 

In [16]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


In [17]:


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


In [18]:
from scraper import fetch_website_contents

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

def stream_gpt(prompt):
    messages = [{"role":"system", "content":system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages, stream=True)
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

def stream_brochure(company_name, url):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    yield from stream_gpt(prompt)



name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
        ["Hugging Face", "https://huggingface.co/"],
        ["DM Academy", "https://www.dm-academy.app/"]
    ], 
    flagging_mode="never"
)
view.launch()


ModuleNotFoundError: No module named 'scraper'